# Avance 4 — Procesamiento de Datos

## Proyecto Inmobiliario — Región Metropolitana

Este notebook implementa cinco procedimientos de procesamiento
sobre los datos del ecosistema Big Data inmobiliario.

### 1. Limpieza y validación de datos

**Objetivo:** identificar registros duplicados, valores faltantes
y problemas de integridad antes de realizar transformaciones
y análisis.

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

# Ruta principal del proyecto
RAIZ = Path.cwd().parent

# Carpeta con los datos originales
RUTA_DATOS = RAIZ / "datalake_crudo" / "chilepropiedades"

# Buscar los archivos JSONL
archivos = sorted(RUTA_DATOS.glob("chilepropiedades_*.jsonl"))

print("=== INVENTARIO DE DATOS ===")
print("Archivos encontrados:", len(archivos))

total_registros = 0

for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:
        cantidad = sum(1 for linea in f if linea.strip())

    total_registros += cantidad

    print(f"{archivo.name}: {cantidad:,} registros")

print("\nTotal de registros:", f"{total_registros:,}")

=== INVENTARIO DE DATOS ===
Archivos encontrados: 14
chilepropiedades_bodega.jsonl: 141 registros
chilepropiedades_departamento.jsonl: 15,043 registros
chilepropiedades_estudio.jsonl: 20 registros
chilepropiedades_galpon.jsonl: 24 registros
chilepropiedades_hotel.jsonl: 4 registros
chilepropiedades_local-comercial.jsonl: 779 registros
chilepropiedades_loft.jsonl: 5 registros
chilepropiedades_masivo.jsonl: 8,615 registros
chilepropiedades_oficina.jsonl: 1,154 registros
chilepropiedades_parcela.jsonl: 1,065 registros
chilepropiedades_sitio.jsonl: 489 registros
chilepropiedades_terreno-agricola.jsonl: 67 registros
chilepropiedades_terreno-industrial.jsonl: 183 registros
chilepropiedades_terreno.jsonl: 1,014 registros

Total de registros: 28,603


In [3]:
import json
from collections import Counter, defaultdict

conteo_urls = Counter()
archivos_por_url = defaultdict(set)

sin_url = 0
json_invalidos = 0

# Recorrer los 14 archivos
for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:

        for numero_linea, linea in enumerate(f, start=1):

            if not linea.strip():
                continue

            try:
                registro = json.loads(linea)

            except json.JSONDecodeError:
                json_invalidos += 1
                continue

            # Obtener y normalizar la URL
            url = registro.get("url")

            if not isinstance(url, str) or not url.strip():
                sin_url += 1
                continue

            url = url.strip().rstrip("/")

            conteo_urls[url] += 1
            archivos_por_url[url].add(archivo.name)

# Calcular duplicados
urls_duplicadas = {
    url: cantidad
    for url, cantidad in conteo_urls.items()
    if cantidad > 1
}

registros_repetidos = sum(
    cantidad - 1
    for cantidad in urls_duplicadas.values()
)

print("=== CONTROL DE DUPLICADOS ===")
print("Registros originales:", total_registros)
print("URLs distintas:", len(conteo_urls))
print("URLs repetidas:", len(urls_duplicadas))
print("Registros duplicados adicionales:", registros_repetidos)
print("Registros sin URL:", sin_url)
print("JSON inválidos:", json_invalidos)

print("\n=== EJEMPLOS DE DUPLICADOS ===")

for url, cantidad in list(urls_duplicadas.items())[:5]:

    print("\nURL:", url)
    print("Apariciones:", cantidad)
    print("Archivos:", sorted(archivos_por_url[url]))

=== CONTROL DE DUPLICADOS ===
Registros originales: 28603
URLs distintas: 28307
URLs repetidas: 296
Registros duplicados adicionales: 296
Registros sin URL: 0
JSON inválidos: 0

=== EJEMPLOS DE DUPLICADOS ===

URL: https://chilepropiedades.cl/ver-publicacion/venta/estacion-central/estudio/avenida-maria-rozas-velasquez-55/110525656
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/santiago/estudio/arturo-prat/130710566
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/san-miguel/estudio/avenida-carlos-valdovinos-metro-franklin/118868378
Apariciones: 2
Archivos: ['chilepropiedades_departamento.jsonl', 'chilepropiedades_estudio.jsonl']

URL: https://chilepropiedades.cl/ver-publicacion/venta/santiago/estudio/garcia-reyes-8/117901234
Apariciones: 2
Archivos: ['chilepropiedades_departa

In [4]:
from collections import defaultdict

# Campos que queremos comparar
campos_comparar = [
    "titulo",
    "precios",
    "caracteristicas",
    "entorno_raw",
    "descripcion_completa"
]

# Recopilar únicamente las publicaciones repetidas
registros_duplicados = defaultdict(list)

for archivo in archivos:

    with open(archivo, "r", encoding="utf-8") as f:

        for linea in f:
            if not linea.strip():
                continue

            registro = json.loads(linea)
            url = registro.get("url", "").strip().rstrip("/")

            if url in urls_duplicadas:
                registros_duplicados[url].append({
                    "archivo": archivo.name,
                    "timestamp": registro.get("timestamp"),
                    **{
                        campo: registro.get(campo)
                        for campo in campos_comparar
                    }
                })

# Comparar el contenido de las publicaciones
identicos = 0
diferentes = 0
ejemplos_diferencias = []

for url, registros in registros_duplicados.items():

    campos_diferentes = []

    for campo in campos_comparar:

        valores = {
            json.dumps(
                registro[campo],
                sort_keys=True,
                ensure_ascii=False
            )
            for registro in registros
        }

        if len(valores) > 1:
            campos_diferentes.append(campo)

    if campos_diferentes:
        diferentes += 1
        ejemplos_diferencias.append({
            "url": url,
            "campos": campos_diferentes,
            "archivos": [r["archivo"] for r in registros]
        })
    else:
        identicos += 1

print("=== COMPARACIÓN DE DUPLICADOS ===")
print("URLs con contenido idéntico:", identicos)
print("URLs con diferencias:", diferentes)

print("\n=== EJEMPLOS DE DIFERENCIAS ===")

for ejemplo in ejemplos_diferencias[:5]:
    print("\nURL:", ejemplo["url"])
    print("Campos diferentes:", ejemplo["campos"])
    print("Archivos:", ejemplo["archivos"])

=== COMPARACIÓN DE DUPLICADOS ===
URLs con contenido idéntico: 296
URLs con diferencias: 0

=== EJEMPLOS DE DIFERENCIAS ===


In [5]:
import json
import hashlib
from datetime import datetime, timezone

# Carpeta de salida
RUTA_SILVER = RAIZ / "datalake_procesado" / "silver"
RUTA_SILVER.mkdir(parents=True, exist_ok=True)

archivo_silver = RUTA_SILVER / "chilepropiedades_unicas.jsonl"
archivo_reporte = RUTA_SILVER / "reporte_deduplicacion.json"

urls_vistas = set()

registros_leidos = 0
registros_conservados = 0
duplicados_eliminados = 0

# Crear el nuevo dataset sin modificar los originales
with open(archivo_silver, "w", encoding="utf-8") as salida:

    for archivo in archivos:

        with open(archivo, "r", encoding="utf-8") as entrada:

            for linea in entrada:

                if not linea.strip():
                    continue

                registro = json.loads(linea)
                registros_leidos += 1

                url = registro["url"].strip().rstrip("/")

                # Omitir publicaciones ya encontradas
                if url in urls_vistas:
                    duplicados_eliminados += 1
                    continue

                urls_vistas.add(url)

                # Identificador estable basado en la URL
                id_estable = hashlib.sha256(
                    url.encode("utf-8")
                ).hexdigest()

                registro["url"] = url
                registro["id_publicacion_estable"] = f"CHP-{id_estable}"
                registro["archivo_origen"] = archivo.name

                salida.write(
                    json.dumps(registro, ensure_ascii=False) + "\n"
                )

                registros_conservados += 1

# Reporte de procesamiento
reporte = {
    "procesamiento": "Deduplicación por URL",
    "fecha_procesamiento": datetime.now(
        timezone.utc
    ).isoformat(),
    "archivos_entrada": len(archivos),
    "registros_originales": registros_leidos,
    "registros_conservados": registros_conservados,
    "duplicados_eliminados": duplicados_eliminados,
    "porcentaje_duplicados": round(
        duplicados_eliminados / registros_leidos * 100, 3
    ),
    "criterio": "Conservar la primera aparición por URL normalizada"
}

with open(archivo_reporte, "w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=4)

# Validar resultado
assert registros_leidos == (
    registros_conservados + duplicados_eliminados
)

assert registros_conservados == len(urls_vistas)

print("=== RESULTADO DE DEDUPLICACIÓN ===")
print("Registros originales:", registros_leidos)
print("Registros conservados:", registros_conservados)
print("Duplicados eliminados:", duplicados_eliminados)
print("Porcentaje eliminado:", reporte["porcentaje_duplicados"], "%")

print("\nDataset Silver:", archivo_silver)
print("Reporte:", archivo_reporte)

=== RESULTADO DE DEDUPLICACIÓN ===
Registros originales: 28603
Registros conservados: 28307
Duplicados eliminados: 296
Porcentaje eliminado: 1.035 %

Dataset Silver: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\chilepropiedades_unicas.jsonl
Reporte: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\reporte_deduplicacion.json


### Procesamiento 1 — Limpieza y deduplicación de datos

**Objetivo:** identificar y eliminar registros repetidos entre los 14 archivos JSONL de Chile Propiedades, evitando contabilizar una misma publicación más de una vez.

**Metodología:** se utilizó la URL normalizada como criterio de identificación. Antes de eliminar los duplicados, se compararon los campos `titulo`, `precios`, `caracteristicas`, `entorno_raw` y `descripcion_completa` para comprobar si presentaban diferencias.

**Resultados:**

| Indicador | Resultado |
|---|---:|
| Registros originales | 28.603 |
| URLs distintas | 28.307 |
| Registros duplicados adicionales | 296 |
| Porcentaje eliminado | 1,035 % |
| Registros sin URL | 0 |
| JSON inválidos | 0 |

Se generó el archivo `chilepropiedades_unicas.jsonl` con las publicaciones deduplicadas y el reporte `reporte_deduplicacion.json` con las métricas del procesamiento.

**Conclusión:** la deduplicación permitió reducir la redundancia y obtener una base de publicaciones con URLs distintas. Los archivos originales se conservaron sin modificaciones, manteniendo la trazabilidad entre los datos crudos y los procesados. Esta operación constituye una primera etapa de limpieza, no una validación completa de la calidad de los datos.

In [6]:
import pandas as pd
import json
from urllib.parse import urlparse

filas = []

# Leer el dataset deduplicado
with open(archivo_silver, "r", encoding="utf-8") as f:

    for linea in f:
        if not linea.strip():
            continue

        registro = json.loads(linea)

        precios = registro.get("precios") or {}
        caracteristicas = registro.get("caracteristicas") or {}

        # Extraer tipo y comuna desde la URL
        partes = urlparse(registro["url"]).path.strip("/").split("/")

        comuna = partes[2] if len(partes) >= 4 else None
        tipo = partes[3] if len(partes) >= 4 else None

        filas.append({
            "id_publicacion": registro["id_publicacion_estable"],
            "url": registro["url"],
            "titulo": registro.get("titulo"),
            "tipo_propiedad": tipo,
            "comuna_slug": comuna,
            "precio_clp": precios.get("clp"),
            "precio_uf": precios.get("uf"),
            "superficie_m2": caracteristicas.get("superficie_m2"),
            "habitaciones": caracteristicas.get("habitaciones"),
            "banos": caracteristicas.get("banos"),
            "fecha_extraccion": registro.get("timestamp"),
            "archivo_origen": registro.get("archivo_origen")
        })

# Crear DataFrame
df_normalizado = pd.DataFrame(filas)

# Estandarizar tipos numéricos
columnas_numericas = [
    "precio_clp",
    "precio_uf",
    "superficie_m2",
    "habitaciones",
    "banos"
]

for columna in columnas_numericas:
    df_normalizado[columna] = pd.to_numeric(
        df_normalizado[columna],
        errors="coerce"
    )

# Los precios y superficies deben ser positivos
for columna in ["precio_clp", "precio_uf", "superficie_m2"]:
    df_normalizado.loc[
        df_normalizado[columna] <= 0, columna
    ] = float("nan")

# Convertir fechas
df_normalizado["fecha_extraccion"] = pd.to_datetime(
    df_normalizado["fecha_extraccion"],
    errors="coerce"
)

print("=== TRANSFORMACIÓN Y NORMALIZACIÓN ===")
print("Registros:", len(df_normalizado))
print("Columnas:", len(df_normalizado.columns))

print("\n=== TIPOS DE DATOS ===")
print(df_normalizado.dtypes)

print("\n=== VISTA PREVIA ===")
display(df_normalizado.head())

=== TRANSFORMACIÓN Y NORMALIZACIÓN ===
Registros: 28307
Columnas: 12

=== TIPOS DE DATOS ===
id_publicacion                 str
url                            str
titulo                         str
tipo_propiedad                 str
comuna_slug                    str
precio_clp                 float64
precio_uf                  float64
superficie_m2              float64
habitaciones               float64
banos                      float64
fecha_extraccion    datetime64[us]
archivo_origen                 str
dtype: object

=== VISTA PREVIA ===


,id_publicacion,url,titulo,tipo_propiedad,comuna_slug,precio_clp,precio_uf,superficie_m2,habitaciones,banos,fecha_extraccion,archivo_origen
0,CHP-5a93ff10dc41edee9a20fc68adf5f6dcef1be4ee52...,https://chilepropiedades.cl/ver-publicacion/ve...,"Bodega en Venta en Carvajal 0330, La Cisterna",bodega,la-cisterna,4531984.0,110.0,NaN,NaN,NaN,2026-10-23 00:02:41,chilepropiedades_bodega.jsonl
1,CHP-8fc606d4bc2d0e41a7e1cb4a2e04ed5569019a010a...,https://chilepropiedades.cl/ver-publicacion/ve...,Bodega en Venta en los Leones,bodega,providencia,8198770.0,199.0,2.0,NaN,NaN,2026-10-23 00:02:42,chilepropiedades_bodega.jsonl
2,CHP-e6aff74106fa592c2797aba033d0d3d514f7461389...,https://chilepropiedades.cl/ver-publicacion/ve...,En Vitacura,bodega,vitacura,22659918.0,550.0,5.0,NaN,NaN,2026-10-23 00:02:43,chilepropiedades_bodega.jsonl
3,CHP-4b7cde6f157da3c63d473e23c80662b50b15715b99...,https://chilepropiedades.cl/ver-publicacion/ve...,En Independencia,bodega,independencia,366266666.0,8890.0,603.0,NaN,NaN,2026-10-23 00:02:43,chilepropiedades_bodega.jsonl
4,CHP-3bf5a423d5f625caa50fa017cf59498b070dccdfb8...,https://chilepropiedades.cl/ver-publicacion/ve...,"Bodega en venta en Buzo Sobenes, Estación Cent...",bodega,estacion-central,407878515.0,9900.0,217.0,NaN,NaN,2026-10-23 00:02:44,chilepropiedades_bodega.jsonl


In [7]:
print("=== DIAGNÓSTICO DE CALIDAD ===")

# Valores faltantes por columna
faltantes = pd.DataFrame({
    "faltantes": df_normalizado.isna().sum(),
    "porcentaje": (
        df_normalizado.isna().mean() * 100
    ).round(2)
})

display(faltantes)

# Revisar superficies sospechosamente pequeñas
superficies_pequenas = df_normalizado[
    df_normalizado["superficie_m2"].notna()
    & (df_normalizado["superficie_m2"] < 10)
]

print("\n=== SUPERFICIES MENORES A 10 M² ===")
print("Registros encontrados:", len(superficies_pequenas))

display(
    superficies_pequenas[
        ["tipo_propiedad", "titulo", "superficie_m2"]
    ].head(10)
)

# Verificar conservación de registros e IDs
print("\n=== VALIDACIÓN DE INTEGRIDAD ===")
print("Registros transformados:", len(df_normalizado))
print(
    "IDs duplicados:",
    df_normalizado["id_publicacion"].duplicated().sum()
)
print(
    "Tipos de propiedad faltantes:",
    df_normalizado["tipo_propiedad"].isna().sum()
)

=== DIAGNÓSTICO DE CALIDAD ===


,faltantes,porcentaje
id_publicacion,0,0.00
url,0,0.00
titulo,0,0.00
tipo_propiedad,0,0.00
comuna_slug,0,0.00
precio_clp,0,0.00
precio_uf,9,0.03
superficie_m2,480,1.70
habitaciones,1698,6.00
banos,1680,5.93



=== SUPERFICIES MENORES A 10 M² ===
Registros encontrados: 113


,tipo_propiedad,titulo,superficie_m2
1,bodega,Bodega en Venta en los Leones,2.0
2,bodega,En Vitacura,5.0
20,bodega,Bodega Departamento,2.0
21,bodega,Bodega en Venta en Peru 980,3.0
25,bodega,Bodegas en Venta en Estación Central,2.0
26,bodega,En Providencia,6.0
27,bodega,Local Comercial Persa Lo Ovalle,2.0
42,bodega,"Bodega en venta en Avenida Las Condes 10415, L...",4.0
43,bodega,Bodegas en Venta en Providencia - Ricardo Lyon...,4.0
49,bodega,"Bodega en venta en Las violetas 2152, Providencia",4.0



=== VALIDACIÓN DE INTEGRIDAD ===
Registros transformados: 28307
IDs duplicados: 0
Tipos de propiedad faltantes: 0


In [8]:
import json
from datetime import datetime, timezone

# Archivos de salida
archivo_parquet = RUTA_SILVER / "chilepropiedades_normalizado.parquet"
archivo_reporte_2 = RUTA_SILVER / "reporte_transformacion.json"

# Guardar dataset transformado
df_normalizado.to_parquet(
    archivo_parquet,
    engine="pyarrow",
    index=False
)

# Generar reporte
reporte_transformacion = {
    "procesamiento": "Transformación y normalización",
    "fecha_procesamiento_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "registros_entrada": 28307,
    "registros_salida": len(df_normalizado),
    "columnas": len(df_normalizado.columns),
    "transformaciones": [
        "Aplanamiento de estructuras JSON anidadas",
        "Extracción de tipo de propiedad y comuna desde URL",
        "Conversión de variables a tipos numéricos",
        "Conversión de fechas a datetime",
        "Marcado de precios y superficies no positivos como faltantes"
    ],
    "valores_faltantes": {
        columna: int(cantidad)
        for columna, cantidad in df_normalizado.isna().sum().items()
    },
    "superficies_menores_10m2": int(
        (
            df_normalizado["superficie_m2"].notna()
            & (df_normalizado["superficie_m2"] < 10)
        ).sum()
    ),
    "ids_duplicados": int(
        df_normalizado["id_publicacion"].duplicated().sum()
    )
}

# Guardar reporte
with open(archivo_reporte_2, "w", encoding="utf-8") as f:
    json.dump(
        reporte_transformacion,
        f,
        ensure_ascii=False,
        indent=4
    )

# Verificar archivo generado
df_verificacion = pd.read_parquet(archivo_parquet)

assert len(df_verificacion) == len(df_normalizado)
assert df_verificacion["id_publicacion"].is_unique

print("=== RESULTADO DEL PROCESAMIENTO 2 ===")
print("Registros guardados:", len(df_verificacion))
print("Columnas:", len(df_verificacion.columns))
print("IDs duplicados:", reporte_transformacion["ids_duplicados"])
print("Superficies menores a 10 m²:",
      reporte_transformacion["superficies_menores_10m2"])

print("\nDataset:", archivo_parquet)
print("Reporte:", archivo_reporte_2)

=== RESULTADO DEL PROCESAMIENTO 2 ===
Registros guardados: 28307
Columnas: 12
IDs duplicados: 0
Superficies menores a 10 m²: 113

Dataset: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\chilepropiedades_normalizado.parquet
Reporte: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\reporte_transformacion.json


### Procesamiento 2 — Transformación y normalización de datos

**Objetivo:** transformar los registros inmobiliarios almacenados en JSONL a una estructura tabular que facilite su procesamiento, almacenamiento y análisis posterior.

**Metodología:** se realizó el aplanamiento de estructuras JSON anidadas, separando las variables de precios y características de los inmuebles. Además, se extrajeron el tipo de propiedad y la comuna desde las URLs, se estandarizaron los tipos de datos numéricos y se convirtieron las fechas al formato `datetime`.

Los valores no positivos de precios y superficies se marcaron como faltantes (`NaN`). Posteriormente, se realizó un diagnóstico de calidad para identificar información ausente y superficies potencialmente sospechosas.

**Resultados:**

| Indicador | Resultado |
|---|---:|
| Registros de entrada | 28.307 |
| Registros transformados | 28.307 |
| Columnas estructuradas | 12 |
| Identificadores duplicados | 0 |
| Precios UF faltantes | 9 |
| Superficies faltantes | 480 |
| Superficies inferiores a 10 m² | 113 |

**Archivos generados:**

- `chilepropiedades_normalizado.parquet`: dataset con las variables estructuradas y los tipos de datos normalizados.
- `reporte_transformacion.json`: reporte de las transformaciones realizadas y los principales indicadores de calidad.

**Conclusión:** la transformación permitió convertir información semiestructurada en un dataset tabular de 28.307 registros, facilitando su utilización en herramientas analíticas. Se identificaron valores faltantes y superficies que requieren revisión, conservando los registros originales para evitar pérdidas de información. El resultado constituye una versión tabular inicial de la capa Silver, pendiente de validaciones adicionales.

---

## Procesamiento tipo 8 — Analítico en memoria con Polars

### Objetivo

Implementar un procesamiento analítico utilizando Polars para obtener estadísticas del mercado inmobiliario de la Región Metropolitana, agrupando las publicaciones por comuna y tipo de propiedad.

### Tecnología utilizada

- **Polars:** biblioteca de procesamiento de datos con un motor analítico columnar.
- **Lazy Evaluation:** permite optimizar las operaciones antes de ejecutarlas.
- **Parquet:** formato columnar utilizado para leer y almacenar los resultados.

### Metodología

1. Leer el dataset normalizado mediante `pl.scan_parquet()`.
2. Agrupar las publicaciones por tipo de propiedad y comuna.
3. Calcular la cantidad de publicaciones de cada grupo.
4. Calcular las medianas de precios en CLP, UF y superficie.
5. Ejecutar el procesamiento mediante `collect()`.
6. Validar los resultados y almacenarlos en formato Parquet.

In [15]:
import polars as pl

# Dataset generado durante el procesamiento anterior
ruta_parquet = (
    RAIZ
    / "datalake_procesado"
    / "silver"
    / "chilepropiedades_normalizado.parquet"
)

# Lectura diferida del archivo Parquet
datos_polars = pl.scan_parquet(ruta_parquet)

# Agregación por comuna y tipo de propiedad
consulta = (
    datos_polars
    .group_by(["tipo_propiedad", "comuna_slug"])
    .agg(
        pl.len().alias("total_publicaciones"),
        pl.col("precio_clp").median().alias("mediana_precio_clp"),
        pl.col("precio_uf").median().alias("mediana_precio_uf"),
        pl.col("superficie_m2").median().alias("mediana_superficie_m2")
    )
    .sort("total_publicaciones", descending=True)
)

# Ejecutar el procesamiento
resultado_polars = consulta.collect()

print("=== PROCESAMIENTO ANALÍTICO CON POLARS ===")
print("Grupos generados:", resultado_polars.height)

display(resultado_polars.head(10))

print("\n=== VALIDACIÓN ===")
print(
    "Publicaciones contabilizadas:",
    resultado_polars["total_publicaciones"].sum()
)

=== PROCESAMIENTO ANALÍTICO CON POLARS ===
Grupos generados: 429


tipo_propiedad,comuna_slug,total_publicaciones,mediana_precio_clp,mediana_precio_uf,mediana_superficie_m2
str,str,u32,f64,f64,f64
"""departamento""","""santiago""",4595,9.4759655e7,2265.0,42.0
"""departamento""","""estacion-central""",1482,7.6209908e7,1849.76,37.0
"""departamento""","""nunoa""",1307,1.91579302e8,4620.0,61.0
"""departamento""","""las-condes""",1068,3.83158605e8,9300.0,106.0
"""departamento""","""san-miguel""",990,1.1535958e8,2800.0,50.0
"""casa""","""colina""",789,5.72604245e8,13900.0,200.0
"""casa""","""puente-alto""",775,1.28e8,3105.99,88.0
"""departamento""","""independencia""",764,8.1416918e7,1950.0,42.0
"""departamento""","""la-florida""",720,1.21539558e8,2912.63,54.0



=== VALIDACIÓN ===
Publicaciones contabilizadas: 28307


In [16]:
import json
from datetime import datetime, timezone

# Definir los archivos de salida
archivo_polars = RUTA_SILVER / "chilepropiedades_agregado_polars.parquet"
archivo_reporte_polars = RUTA_SILVER / "reporte_polars.json"

# Validar que no se perdieron publicaciones
total_entrada = (
    pl.scan_parquet(ruta_parquet)
    .select(pl.len())
    .collect()
    .item()
)

total_agrupado = int(
    resultado_polars["total_publicaciones"].sum()
)

assert total_entrada == total_agrupado, (
    "La cantidad de publicaciones no coincide."
)

# Guardar las estadísticas agregadas
resultado_polars.write_parquet(archivo_polars)

# Crear reporte del procesamiento
reporte_polars = {
    "procesamiento": "Analítico en memoria",
    "tecnologia": "Polars",
    "tipo_pauta": 8,
    "fecha_procesamiento_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "registros_entrada": total_entrada,
    "publicaciones_contabilizadas": total_agrupado,
    "grupos_generados": resultado_polars.height,
    "dimensiones": [
        "tipo_propiedad",
        "comuna_slug"
    ],
    "metricas": [
        "total_publicaciones",
        "mediana_precio_clp",
        "mediana_precio_uf",
        "mediana_superficie_m2"
    ],
    "modalidad": "Lazy Evaluation sobre Parquet"
}

with open(archivo_reporte_polars, "w", encoding="utf-8") as f:
    json.dump(
        reporte_polars,
        f,
        ensure_ascii=False,
        indent=4
    )

# Comprobar el archivo generado
verificacion = pl.read_parquet(archivo_polars)

assert verificacion.height == resultado_polars.height

print("=== RESULTADO DEL PROCESAMIENTO POLARS ===")
print("Registros de entrada:", total_entrada)
print("Publicaciones contabilizadas:", total_agrupado)
print("Grupos generados:", verificacion.height)
print("Columnas generadas:", len(verificacion.columns))

print("\nDataset:", archivo_polars)
print("Reporte:", archivo_reporte_polars)

=== RESULTADO DEL PROCESAMIENTO POLARS ===
Registros de entrada: 28307
Publicaciones contabilizadas: 28307
Grupos generados: 429
Columnas generadas: 6

Dataset: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\chilepropiedades_agregado_polars.parquet
Reporte: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_procesado\silver\reporte_polars.json



### Resultados

Se implementó un procesamiento analítico en memoria utilizando Polars 2.1.0, aplicando ejecución diferida (*Lazy Evaluation*) sobre un dataset inmobiliario en formato Parquet.

| Indicador | Resultado |
|---|---:|
| Registros procesados | 28.307 |
| Grupos generados | 429 |
| Publicaciones contabilizadas | 28.307 |
| Variables de agrupación | 2 |
| Métricas calculadas | 4 |
| Formato de salida | Parquet |

**Indicadores calculados:**

- Cantidad de publicaciones por comuna y tipo de propiedad.
- Mediana de precios en pesos chilenos (CLP).
- Mediana de precios en UF.
- Mediana de superficie en metros cuadrados.

**Archivos generados:**

- `chilepropiedades_agregado_polars.parquet`: estadísticas agrupadas por comuna y tipo de propiedad.
- `reporte_polars.json`: reporte de ejecución y validación.

### Conclusión

Se implementó un procesamiento analítico con Polars que permitió transformar 28.307 publicaciones inmobiliarias en 429 grupos estadísticos.

La validación confirmó que todas las publicaciones fueron contabilizadas durante la agregación, sin pérdidas de registros.

Los resultados constituyen indicadores exploratorios que podrán utilizarse posteriormente en el Data Warehouse, Data Lakehouse y las herramientas de visualización.

Las estadísticas deberán interpretarse considerando las anomalías y limitaciones de calidad identificadas durante el preprocesamiento.


---

## Procesamiento tipo 1 — Batch distribuido con Apache Spark

### Objetivo

Implementar un procesamiento por lotes utilizando Apache Spark
(PySpark) para transformar y validar publicaciones inmobiliarias
provenientes de Chile Propiedades.

### Tecnología utilizada

- **Apache Spark:** motor de procesamiento distribuido de datos.
- **PySpark:** interfaz de Python para utilizar Apache Spark.
- **Parquet:** formato columnar para almacenar los resultados.

### Metodología

1. Leer los archivos JSONL originales de Chile Propiedades.
2. Validar la estructura y los campos principales.
3. Identificar y eliminar publicaciones duplicadas.
4. Transformar los datos a una estructura tabular.
5. Guardar los resultados procesados en formato Parquet.
6. Comprobar la integridad del dataset generado.

### Resultado esperado

Obtener un dataset estructurado y deduplicado mediante
Apache Spark, que pueda utilizarse posteriormente en el
Data Lakehouse y en los análisis inmobiliarios.

In [18]:
import subprocess

try:
    resultado = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True
    )

    print("=== VERIFICACIÓN DE JAVA ===")
    print(resultado.stderr or resultado.stdout)
    print("Código de salida:", resultado.returncode)

except FileNotFoundError:
    print("Java no está disponible en el PATH del entorno.")

=== VERIFICACIÓN DE JAVA ===
java version "21.0.12" 2026-07-21 LTS
Java(TM) SE Runtime Environment (build 21.0.12+7-LTS-205)
Java HotSpot(TM) 64-Bit Server VM (build 21.0.12+7-LTS-205, mixed mode, sharing)

Código de salida: 0


In [21]:
import pyspark

print("=== VERIFICACIÓN DE PYSPARK ===")
print("Versión instalada:", pyspark.__version__)

=== VERIFICACIÓN DE PYSPARK ===
Versión instalada: 4.0.1


In [22]:
import os
import sys
from pyspark.sql import SparkSession

# Utilizar el mismo Python que ejecuta Jupyter
os.environ["PYSPARK_PYTHON"] = sys.executable

# Crear sesión de Apache Spark
spark = (
    SparkSession.builder
    .appName("Proyecto_Inmobiliario_Batch")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Reducir mensajes informativos
spark.sparkContext.setLogLevel("ERROR")

print("=== VERIFICACIÓN DE APACHE SPARK ===")
print("Versión de Spark:", spark.version)
print("Modo de ejecución:", spark.sparkContext.master)

# Comprobar que el motor puede ejecutar una operación
prueba = spark.range(100)

print("Registros de prueba:", prueba.count())
print("Spark funcionando correctamente.")

=== VERIFICACIÓN DE APACHE SPARK ===
Versión de Spark: 4.0.1
Modo de ejecución: local[2]
Registros de prueba: 100
Spark funcionando correctamente.


In [23]:
from pathlib import Path

# Ubicación de los archivos originales
ruta_datos_spark = (
    RAIZ / "datalake_crudo" / "chilepropiedades"
)

# Identificar los 14 archivos JSONL
archivos_spark = sorted(
    ruta_datos_spark.glob("chilepropiedades_*.jsonl")
)

# Convertir rutas al formato URI para Spark
rutas_jsonl = [
    archivo.resolve().as_uri()
    for archivo in archivos_spark
]

print("=== LECTURA BATCH CON APACHE SPARK ===")
print("Archivos encontrados:", len(rutas_jsonl))

# Leer los archivos mediante Spark
df_spark = (
    spark.read
    .option("mode", "PERMISSIVE")
    .option("multiLine", "false")
    .json(rutas_jsonl)
)

# Verificar registros
total_spark = df_spark.count()

print("Registros cargados:", total_spark)

# Mostrar estructura
print("\n=== ESQUEMA DE LOS DATOS ===")
df_spark.printSchema()

# Mostrar una muestra
print("\n=== VISTA PREVIA ===")
df_spark.select(
    "id_propiedad",
    "url",
    "titulo",
    "precios",
    "caracteristicas"
).show(3, truncate=50)

=== LECTURA BATCH CON APACHE SPARK ===
Archivos encontrados: 14
Registros cargados: 28603

=== ESQUEMA DE LOS DATOS ===
root
 |-- caracteristicas: struct (nullable = true)
 |    |-- banos: long (nullable = true)
 |    |-- habitaciones: long (nullable = true)
 |    |-- superficie_m2: long (nullable = true)
 |-- descripcion_completa: string (nullable = true)
 |-- entorno_raw: struct (nullable = true)
 |    |-- colegios: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- direccion: string (nullable = true)
 |    |    |    |-- distancia_metros: long (nullable = true)
 |    |    |    |-- nombre: string (nullable = true)
 |    |-- farmacias: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- direccion: string (nullable = true)
 |    |    |    |-- distancia_metros: long (nullable = true)
 |    |    |    |-- nombre: string (nullable = true)
 |    |-- metro y tren: array (nullable = true)
 |    |    |-- 

In [24]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Normalizar URLs y convertir las fechas
df_spark_preparado = (
    df_spark
    .withColumn(
        "url_normalizada",
        F.regexp_replace(
            F.trim(F.col("url")),
            r"/+$",
            ""
        )
    )
    .withColumn(
        "fecha_extraccion_dt",
        F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss")
    )
)

# Verificar que las publicaciones tengan URL
sin_url = df_spark_preparado.filter(
    F.col("url_normalizada").isNull()
    | (F.length("url_normalizada") == 0)
).count()

if sin_url > 0:
    raise ValueError(
        f"Se encontraron {sin_url} registros sin URL."
    )

# Definir el criterio para conservar una publicación
ventana = Window.partitionBy(
    "url_normalizada"
).orderBy(
    F.col("fecha_extraccion_dt").desc_nulls_last(),
    F.col("id_propiedad").asc_nulls_last()
)

# Deduplicación mediante funciones de ventana
df_spark_unicos = (
    df_spark_preparado
    .withColumn(
        "numero_fila",
        F.row_number().over(ventana)
    )
    .filter(F.col("numero_fila") == 1)
    .drop("numero_fila")
)

# Validar resultados
total_original = df_spark.count()
total_unicos = df_spark_unicos.count()
duplicados_eliminados = total_original - total_unicos

print("=== DEDUPLICACIÓN CON APACHE SPARK ===")
print("Registros originales:", total_original)
print("Registros únicos:", total_unicos)
print("Duplicados eliminados:", duplicados_eliminados)
print("Registros sin URL:", sin_url)

assert total_original == total_unicos + duplicados_eliminados

print("\nValidación completada.")

=== DEDUPLICACIÓN CON APACHE SPARK ===
Registros originales: 28603
Registros únicos: 28307
Duplicados eliminados: 296
Registros sin URL: 0

Validación completada.


In [25]:
from pyspark.sql import functions as F

# Transformar los registros deduplicados
df_spark_tabular = (
    df_spark_unicos
    .select(
        # Identificación y trazabilidad
        F.concat(
            F.lit("CHP-"),
            F.sha2(F.col("url_normalizada"), 256)
        ).alias("id_publicacion"),

        F.lit("chilepropiedades").alias("fuente"),

        F.col("url_normalizada").alias("url"),
        F.col("titulo"),

        # Ubicación y tipo desde la URL
        F.regexp_extract(
            "url_normalizada",
            r"/ver-publicacion/[^/]+/[^/]+/([^/]+)/",
            1
        ).alias("tipo_propiedad"),

        F.regexp_extract(
            "url_normalizada",
            r"/ver-publicacion/[^/]+/([^/]+)/",
            1
        ).alias("comuna_slug"),

        # Variables numéricas
        F.when(
            F.col("precios.clp") > 0,
            F.col("precios.clp")
        ).alias("precio_clp"),

        F.when(
            F.col("precios.uf") > 0,
            F.col("precios.uf")
        ).alias("precio_uf"),

        F.when(
            F.col("caracteristicas.superficie_m2") > 0,
            F.col("caracteristicas.superficie_m2")
        ).alias("superficie_m2"),

        F.col("caracteristicas.habitaciones").alias(
            "habitaciones"
        ),

        F.col("caracteristicas.banos").alias("banos"),

        # Información adicional
        F.col("fecha_extraccion_dt").alias(
            "fecha_extraccion"
        ),

        F.col("descripcion_completa"),
        F.col("entorno_raw")
    )
)

print("=== TRANSFORMACIÓN BATCH CON PYSPARK ===")

print("Registros transformados:", df_spark_tabular.count())
print("Columnas generadas:", len(df_spark_tabular.columns))

print("\n=== ESTRUCTURA TABULAR ===")
df_spark_tabular.printSchema()

print("\n=== VISTA PREVIA ===")
df_spark_tabular.select(
    "tipo_propiedad",
    "comuna_slug",
    "precio_clp",
    "precio_uf",
    "superficie_m2"
).show(5)

=== TRANSFORMACIÓN BATCH CON PYSPARK ===
Registros transformados: 28307
Columnas generadas: 14

=== ESTRUCTURA TABULAR ===
root
 |-- id_publicacion: string (nullable = true)
 |-- fuente: string (nullable = false)
 |-- url: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- tipo_propiedad: string (nullable = true)
 |-- comuna_slug: string (nullable = true)
 |-- precio_clp: long (nullable = true)
 |-- precio_uf: double (nullable = true)
 |-- superficie_m2: long (nullable = true)
 |-- habitaciones: long (nullable = true)
 |-- banos: long (nullable = true)
 |-- fecha_extraccion: timestamp (nullable = true)
 |-- descripcion_completa: string (nullable = true)
 |-- entorno_raw: struct (nullable = true)
 |    |-- colegios: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- direccion: string (nullable = true)
 |    |    |    |-- distancia_metros: long (nullable = true)
 |    |    |    |-- nombre: string (nullable = true)
 |    |-- f

In [26]:
from pyspark.sql import functions as F

# Indicadores de calidad
diagnostico_spark = df_spark_tabular.agg(

    F.count("*").alias("total_registros"),

    F.countDistinct("id_publicacion").alias("ids_distintos"),

    F.sum(
        F.when(F.col("precio_clp").isNull(), 1).otherwise(0)
    ).alias("sin_precio_clp"),

    F.sum(
        F.when(F.col("precio_uf").isNull(), 1).otherwise(0)
    ).alias("sin_precio_uf"),

    F.sum(
        F.when(F.col("superficie_m2").isNull(), 1).otherwise(0)
    ).alias("sin_superficie"),

    F.sum(
        F.when(F.col("superficie_m2") < 10, 1).otherwise(0)
    ).alias("superficies_menores_10"),

    F.sum(
        F.when(F.col("precio_uf") > 1_000_000, 1).otherwise(0)
    ).alias("precios_uf_extremos"),

    F.sum(
        F.when(F.col("fecha_extraccion").isNull(), 1).otherwise(0)
    ).alias("fechas_invalidas")

)

print("=== DIAGNÓSTICO DE CALIDAD CON PYSPARK ===")

diagnostico_spark.show(vertical=True)

# Revisar publicaciones con precios UF extremos
print("\n=== PRECIOS QUE REQUIEREN REVISIÓN ===")

df_spark_tabular.filter(
    F.col("precio_uf") > 1_000_000
).select(
    "tipo_propiedad",
    "comuna_slug",
    "precio_clp",
    "precio_uf",
    "superficie_m2"
).orderBy(
    F.col("precio_uf").desc()
).show(10, truncate=False)

# Validación de integridad
assert df_spark_tabular.count() == 28307

assert (
    df_spark_tabular
    .select("id_publicacion")
    .distinct()
    .count()
) == 28307

print("\nValidación de integridad completada.")

=== DIAGNÓSTICO DE CALIDAD CON PYSPARK ===
-RECORD 0-----------------------
 total_registros        | 28307 
 ids_distintos          | 28307 
 sin_precio_clp         | 0     
 sin_precio_uf          | 9     
 sin_superficie         | 480   
 superficies_menores_10 | 113   
 precios_uf_extremos    | 9     
 fechas_invalidas       | 0     


=== PRECIOS QUE REQUIEREN REVISIÓN ===
+---------------+----------------+-------------+-------------+-------------+
|tipo_propiedad |comuna_slug     |precio_clp   |precio_uf    |superficie_m2|
+---------------+----------------+-------------+-------------+-------------+
|departamento   |santiago        |2966389200000|7.2E7        |43           |
|sitio          |maipu           |2200000000000|5.339825266E7|700          |
|departamento   |estacion-central|1573834270000|3.82E7       |20           |
|local-comercial|lo-barnechea    |1071196100000|2.6E7        |175          |
|parcela        |alhue           |1029996250000|2.5E7        |NULL         |
|of

In [28]:
print("=== VERSIONES DEL ENTORNO ===")

print("Spark:", spark.version)

print(
    "Hadoop:",
    spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

=== VERSIONES DEL ENTORNO ===
Spark: 4.0.1
Hadoop: 3.4.1


In [29]:
import json
import pyarrow.dataset as ds

# Archivos creados por Spark desde Docker
ruta_spark = RUTA_SILVER / "chilepropiedades_batch_spark"
ruta_reporte = RUTA_SILVER / "reporte_pyspark.json"

# Leer el dataset Parquet
dataset_spark = ds.dataset(
    str(ruta_spark),
    format="parquet"
)

# Leer el reporte
with open(ruta_reporte, "r", encoding="utf-8") as f:
    reporte_spark = json.load(f)

# Validaciones
total_registros = dataset_spark.count_rows()
total_columnas = len(dataset_spark.schema.names)

assert total_registros == 28307
assert total_columnas == 14

print("=== VERIFICACIÓN DEL PROCESAMIENTO SPARK ===")
print("Registros:", total_registros)
print("Columnas:", total_columnas)
print("Tecnología:", reporte_spark["tecnologia"])
print("Versión:", reporte_spark["version_spark"])
print("Modo:", reporte_spark["modo_ejecucion"])
print("Duplicados eliminados:", reporte_spark["duplicados_eliminados"])

print("\nVerificación completada correctamente.")

=== VERIFICACIÓN DEL PROCESAMIENTO SPARK ===
Registros: 28307
Columnas: 14
Tecnología: PySpark
Versión: 4.0.1
Modo: local[2]
Duplicados eliminados: 296

Verificación completada correctamente.



### Resultados

Se implementó un procesamiento batch mediante Apache Spark 4.0.1 (PySpark), ejecutado dentro de un contenedor Docker con Linux.

El procesamiento utilizó los 14 archivos JSONL originales de Chile Propiedades, realizando lectura, deduplicación, transformación de estructuras anidadas y validación de integridad.

| Indicador | Resultado |
|---|---:|
| Archivos JSONL procesados | 14 |
| Registros originales | 28.603 |
| Registros conservados | 28.307 |
| Duplicados eliminados | 296 |
| Columnas generadas | 14 |
| Identificadores duplicados en la salida | 0 |
| Modo de ejecución | local[2] |
| Formato de salida | Parquet |

**Controles de calidad realizados:**

- 9 publicaciones sin precio en UF.
- 480 publicaciones sin superficie.
- 113 superficies inferiores a 10 m².
- 9 precios superiores a 1.000.000 UF.
- 0 fechas de extracción inválidas.

Estos resultados corresponden a controles exploratorios. Los valores sospechosos fueron identificados, pero no eliminados automáticamente.

**Archivos generados:**

- `chilepropiedades_batch_spark/`: dataset procesado en formato Parquet.
- `reporte_pyspark.json`: reporte de ejecución y trazabilidad.

### Conclusión

Se implementó exitosamente un procesamiento por lotes con Apache Spark, transformando 28.603 registros originales en un dataset estructurado de 28.307 publicaciones con URLs distintas.

La ejecución mediante Docker permitió guardar los resultados utilizando el motor nativo de Spark. Posteriormente, se verificó desde Jupyter Notebook que el dataset generado podía leerse correctamente.

El procesamiento se ejecutó en modo local con dos hilos (`local[2]`), sin utilizar un clúster multinodo.

El resultado podrá utilizarse posteriormente en el Data Lakehouse y en los procesos de análisis del mercado inmobiliario.



---

## Procesamiento tipo 3 — SQL sobre archivos con Trino

### Objetivo

Implementar consultas SQL directamente sobre archivos Parquet almacenados en el Data Lake, sin necesidad de cargar previamente los registros a una base de datos relacional.

### Tecnología utilizada

- **Trino 484:** motor de consultas SQL distribuido.
- **Hive Metastore 4.0.1:** servicio de gestión de metadatos, respaldado por PostgreSQL.
- **SILO/S3:** almacenamiento de los archivos Parquet generados con PySpark.
- **Docker Compose:** infraestructura para ejecutar los servicios.

### Metodología

1. Almacenar los archivos Parquet procesados por PySpark en el bucket `lake-procesado`.
2. Configurar Hive Metastore con PostgreSQL para administrar los metadatos.
3. Configurar el catálogo `lake` de Trino y su conexión con SILO.
4. Registrar la tabla externa `lake.inmobiliario.publicaciones`.
5. Verificar el acceso a los datos mediante una consulta `COUNT(*)`.
6. Ejecutar consultas SQL de agregación por tipo de propiedad.

### Resultados

Se verificó la lectura de **28.307 publicaciones inmobiliarias** mediante Trino.

Se ejecutaron consultas SQL para calcular la cantidad de publicaciones y la mediana aproximada de precios en UF por categoría.

| Tipo de propiedad | Publicaciones | Mediana aproximada (UF) |
|---|---:|---:|
| Departamento | 15.005 | 2.714,21 |
| Casa | 8.607 | 5.876,34 |
| Oficina | 1.154 | 5.313,14 |
| Parcela | 1.065 | 6.931,23 |
| Local comercial | 779 | 9.009,52 |

Los resultados mostrados corresponden a cinco de las categorías principales. Las estadísticas son exploratorias y requieren validaciones adicionales de precios.

**Archivos SQL utilizados:**

- `sql/02_trino_publicaciones.sql`: definición de la tabla externa.
- `sql/03_trino_analisis.sql`: consultas analíticas sobre los archivos Parquet.

### Conclusión

Se implementó exitosamente un procesamiento SQL sobre archivos utilizando Trino, consultando información inmobiliaria almacenada en SILO mediante una tabla externa.

El procesamiento permitió obtener indicadores por tipo de propiedad sin trasladar los registros al Data Warehouse.

La implementación demuestra la interoperabilidad entre Trino, Hive Metastore, PostgreSQL y el almacenamiento compatible con S3.



---

## Procesamiento tipo 4 — Transformaciones ELT con dbt

### Objetivo

Implementar transformaciones analíticas mediante SQL utilizando dbt (data build tool), aprovechando los datos previamente cargados en PostgreSQL y generando modelos reutilizables para el análisis inmobiliario.

### Tecnología utilizada

- **dbt Core:** herramienta para definir, ejecutar y validar transformaciones SQL.
- **dbt-postgres:** adaptador que permite ejecutar los modelos en PostgreSQL.
- **PostgreSQL:** motor de base de datos que almacena tanto los registros originales como los resultados transformados.
- **SQL:** lenguaje utilizado para realizar deduplicaciones, filtrados, relaciones y agregaciones.

### Metodología

1. Crear el proyecto `dbt_inmobiliario` y configurar su conexión con PostgreSQL mediante variables de entorno.
2. Utilizar las tablas `dw.fact_publicaciones` y `dw.dim_propiedad` como fuentes de información.
3. Implementar el modelo `resumen_publicaciones.sql`.
4. Seleccionar la extracción más reciente de cada publicación utilizando `ROW_NUMBER()`.
5. Filtrar registros con precios válidos y separar las publicaciones según su moneda.
6. Calcular el número de publicaciones, precio promedio y mediana por tipo de propiedad y moneda.
7. Materializar los resultados como una tabla en PostgreSQL.
8. Ejecutar pruebas automáticas de calidad mediante `dbt test`.

### Resultados

Se generó correctamente el modelo analítico `dbt.resumen_publicaciones`, utilizando los 240 registros de la carga inicial del Data Warehouse.

| Tipo de propiedad | Moneda | Publicaciones | Precio promedio | Mediana |
|---|---|---:|---:|---:|
| Casa | CLP | 18 | 263.444.444,44 | 187.500.000 |
| Casa | UF | 30 | 10.074,63 | 8.277 |
| Departamento | CLP | 37 | 121.848.108,11 | 105.000.000 |
| Departamento | UF | 155 | 2.940,32 | 2.390 |

El modelo produjo cuatro grupos analíticos, que representan un total de 240 publicaciones.

Las métricas se calcularon por separado para CLP y UF, evitando combinar precios expresados en unidades monetarias distintas.

### Validación de calidad

Se incorporaron pruebas automáticas mediante dbt para verificar:

- Ausencia de valores nulos en las cinco columnas del modelo.
- Restricción del campo `moneda` a los valores `UF` y `CLP`.

La ejecución de las seis pruebas finalizó exitosamente.

### Conclusión

Se implementó un procesamiento ELT con dbt, ejecutando transformaciones SQL directamente sobre los datos almacenados en PostgreSQL.

El modelo permitió consolidar registros inmobiliarios en indicadores analíticos reutilizables, incorporando pruebas automatizadas de calidad.

Esta implementación constituye el cuarto tipo de procesamiento del ecosistema Big Data.

**Alcance:** esta etapa utiliza la carga inicial de 240 publicaciones del Data Warehouse. La integración del conjunto completo de fuentes inmobiliarias y datos complementarios se desarrollará posteriormente.
